<div style="background-color:#F3F2EE">
    <br /><br />
        <p style="text-align: center;">
            <font size="6" color='#0A1781'>
                <strong>
                    Databricks Credit Risk Lakehouse - Data Science Project
                </strong>
              </font>
        </p>
        <p style="text-align: center;">
            <font size="6" color='#C58A1E'>
                <strong>
                    Raw to Bronze Inestion
                </strong>
            </font>
        </p>
        <p style="text-align: center;">
            <font size="5" color='#C58A1E'>
                <strong>
                    Apresentar o propósito do notebook e posicionar a ingestão<BR />Raw → Bronze dentro da arquitetura Lakehouse.
                </strong>
            </font>
        </p>
    <br />
</div>

<div style="background-color:#F3F2EE">
    <p style="text-align: right;">
      <font size="4" color='#444444'>
            Roberto SSoares - LfLngLrnng
      </font>
    </p>
    <p style="text-align: right;"><font size="2" color='#444444'>
        <a href="https://www.linkedin.com/in/roberto-dos-santos-soares/">in/roberto-dos-santos-soares</a><br /><a href="https://roberto-ssoares.github.io/meu-portfolio/">Portifólio: roberto-ssoares</a>
    </p>
    <p style="text-align: right;">
        <font size="4" color='#444444'>
            " [+] Faturamento [-] Custo [+] Qualidade de vida "
        </font>
        <br />
        <font size="2" color='#918e8e'>"Mestre Bruno Jardim"
        </font>
    </p>        
    <p style="text-align: right;">        
        <font size="2" color='#918e8e'>           
        </font>
    </p>
</div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>📌 Objetivo</strong></font>

<font size="2" color='#66666'>

>- **Ações realizadas**
    - *Leitura dos arquivos CSV da camada Raw.*
    - *Validação da existência dos arquivos de origem.*
    - *Criação de metadados técnicos de ingestão.*
    - *Persistência dos dados na camada Bronze em formato Parquet.*
    - *Geração de manifesto, reconciliação e evidências para atualização futura do Knowledge Graph.*

>- **Justificativa técnica**
    - A camada Bronze representa o primeiro estágio estruturado do Lakehouse.
    - Ela preserva os dados próximos da origem, mas adiciona rastreabilidade técnica, permitindo auditoria, reprocessamento e evolução controlada do pipeline.

>- **Resultados esperados**
    - Ao final deste notebook, teremos tabelas Bronze em formato Parquet para customers, contracts, payments e credit_events, além de artefatos de controle e evidências reais para o KG.

---

</font></div>

In [1]:
#!uv pip install watermark -q -U
#!uv pip install tabulate -q -U

In [1]:
from datetime import date, datetime
from pathlib import Path
import uuid

import pandas as pd
import numpy as np

In [2]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "RobertoSSoares-LfLngLrnng"

Author: RobertoSSoares-LfLngLrnng



In [3]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [4]:
PROJECT_NAME = "Databricks Credit Risk Lakehouse"
NOTEBOOK_NAME = "03_raw_to_bronze_ingestion"
STUDY_DATE = date.today().isoformat()

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
EXPORTS_DIR = DATA_DIR / "exports"
DOCS_DIR = BASE_DIR / "docs"

for directory in [
    RAW_DIR,
    BRONZE_DIR,
    SILVER_DIR,
    GOLD_DIR,
    EXPORTS_DIR,
    DOCS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

{
    "project": PROJECT_NAME,
    "notebook": NOTEBOOK_NAME,
    "study_date": STUDY_DATE,
    "base_dir": str(BASE_DIR),
    "raw_dir": str(RAW_DIR),
    "bronze_dir": str(BRONZE_DIR),
}


{'project': 'Databricks Credit Risk Lakehouse',
 'notebook': '03_raw_to_bronze_ingestion',
 'study_date': '2026-04-29',
 'base_dir': 'D:\\_DS-Projects\\Data-Science\\databricks-credit-risk-lakehouse',
 'raw_dir': 'D:\\_DS-Projects\\Data-Science\\databricks-credit-risk-lakehouse\\data\\raw',
 'bronze_dir': 'D:\\_DS-Projects\\Data-Science\\databricks-credit-risk-lakehouse\\data\\bronze'}

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 1. Estratégia da camada Bronze</strong></font>

<font size="2" color='#66666'>

>- A camada Bronze será criada a partir dos arquivos CSV preservados na camada Raw.

>- Nesta etapa, não vamos fazer limpeza de negócio, deduplicação ou imputação. O objetivo é manter os dados próximos da origem, adicionando apenas metadados técnicos.

>- **Princípios da Bronze**
    - *preservar dados brutos;*
    - *registrar origem do arquivo;*
    - *registrar data/hora de ingestão;*
    - *manter rastreabilidade por entidade;*
    - *persistir em formato analítico;*
    - *permitir reprocessamento idempotente.*

>- **Entidades processadas**
    - customers;
    - contracts;
    - payments;
    - credit_events.

---

</font></div>

In [5]:
RAW_FILES = {
    "customers": RAW_DIR / "customers.csv",
    "contracts": RAW_DIR / "contracts.csv",
    "payments": RAW_DIR / "payments.csv",
    "credit_events": RAW_DIR / "credit_events.csv",
}

BRONZE_OUTPUTS = {
    entity: BRONZE_DIR / f"bronze_{entity}.parquet"
    for entity in RAW_FILES
}

RAW_FILES


{'customers': WindowsPath('D:/_DS-Projects/Data-Science/databricks-credit-risk-lakehouse/data/raw/customers.csv'),
 'contracts': WindowsPath('D:/_DS-Projects/Data-Science/databricks-credit-risk-lakehouse/data/raw/contracts.csv'),
 'payments': WindowsPath('D:/_DS-Projects/Data-Science/databricks-credit-risk-lakehouse/data/raw/payments.csv'),
 'credit_events': WindowsPath('D:/_DS-Projects/Data-Science/databricks-credit-risk-lakehouse/data/raw/credit_events.csv')}

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 2. Validação dos arquivos Raw</strong></font>

<font size="2" color='#66666'>

>- Antes de iniciar a ingestão, precisamos garantir que todos os arquivos esperados existem.

>- Essa checagem evita que o pipeline prossiga com fontes incompletas.

---

</font></div>

In [6]:
raw_file_validation = pd.DataFrame(
    [
        {
            "entity": entity,
            "raw_file": str(path),
            "exists": path.exists(),
            "file_size_bytes": path.stat().st_size if path.exists() else 0,
        }
        for entity, path in RAW_FILES.items()
    ]
)

raw_file_validation


,entity,raw_file,exists,file_size_bytes
0,customers,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\customers.csv,True,46038
1,contracts,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\contracts.csv,True,59008
2,payments,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\payments.csv,True,1213225
3,credit_events,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\credit_events.csv,True,52480


In [7]:
missing_files = raw_file_validation.loc[~raw_file_validation["exists"], "raw_file"].tolist()

if missing_files:
    raise FileNotFoundError(f"Arquivos Raw não encontrados: {missing_files}")

print("Todos os arquivos Raw esperados foram encontrados.")

Todos os arquivos Raw esperados foram encontrados.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 3. Funções auxiliares de ingestão</strong></font>

<font size="2" color='#66666'>

>- Nesta etapa, vamos criar funções reutilizáveis para:
    - *carregar arquivos CSV;*
    - *adicionar metadados técnicos;*
    - *gerar identificador de ingestão;*
    - *gerar hash técnico por registro;*
    - *salvar em Parquet;*
    - *criar sumários de qualidade e reconciliação.*

---

</font></div>

In [8]:
def read_raw_csv(path: Path) -> pd.DataFrame:
    """
    Lê um arquivo CSV da camada Raw.

    A leitura mantém uma abordagem simples e reprodutível.
    A limpeza de negócio será feita apenas nas camadas posteriores.
    """
    return pd.read_csv(path)


def add_bronze_metadata(df: pd.DataFrame, entity: str, source_path: Path, ingestion_id: str) -> pd.DataFrame:
    """
    Adiciona metadados técnicos da camada Bronze.
    """
    bronze_df = df.copy()
    
    bronze_df.insert(0, "bronze_record_id", [f"{entity.upper()}_{i:08d}" for i in range(1, len(bronze_df) + 1)])
    bronze_df["bronze_ingestion_id"] = ingestion_id
    bronze_df["bronze_ingested_at"] = datetime.now().isoformat(timespec="seconds")
    bronze_df["bronze_ingestion_date"] = STUDY_DATE
    bronze_df["bronze_source_entity"] = entity
    bronze_df["bronze_source_file"] = source_path.name
    bronze_df["bronze_source_path"] = str(source_path)
    bronze_df["bronze_source_layer"] = "raw"
    bronze_df["bronze_target_layer"] = "bronze"
    bronze_df["bronze_pipeline_step"] = NOTEBOOK_NAME
    
    # Hash técnico do registro para rastreabilidade.
    source_cols = [col for col in df.columns]
    bronze_df["bronze_row_hash"] = pd.util.hash_pandas_object(
        df[source_cols].astype(str),
        index=False
    ).astype(str)
    
    return bronze_df


def write_bronze_parquet(df: pd.DataFrame, output_path: Path) -> None:
    """
    Escreve DataFrame em Parquet na camada Bronze.

    O overwrite é intencional nesta fase para garantir reexecução idempotente.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, index=False)


def profile_dataframe(entity: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera perfil básico por coluna.
    """
    rows = []
    total_rows = len(df)
    
    for column in df.columns:
        null_count = int(df[column].isna().sum())
        blank_count = int((df[column].astype(str).str.strip() == "").sum())
        
        rows.append(
            {
                "entity": entity,
                "column": column,
                "total_rows": total_rows,
                "null_count": null_count,
                "blank_count": blank_count,
                "null_pct": round(null_count / total_rows, 4) if total_rows > 0 else 0,
                "blank_pct": round(blank_count / total_rows, 4) if total_rows > 0 else 0,
                "dtype": str(df[column].dtype),
                "sample_values": ", ".join(df[column].dropna().astype(str).head(3).tolist()),
            }
        )
    
    return pd.DataFrame(rows)


def df_to_markdown_safe(df: pd.DataFrame) -> str:
    """
    Converte DataFrame para Markdown.
    Caso tabulate não esteja disponível, usa string simples.
    """
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)
        

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 4. Execução da ingestão Raw → Bronze</strong></font>

<font size="2" color='#66666'>

>- Agora vamos executar a ingestão das quatro entidades.

>- Para cada entidade:
    1. ler o CSV da Raw;
    2. criar metadados técnicos;
    3. persistir em Parquet na Bronze;
    4. registrar métricas de controle.

---

</font></div>

In [9]:
ingestion_id = f"INGEST_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

bronze_tables = {}
ingestion_manifest_rows = []
reconciliation_rows = []
schema_profiles = []
quality_profiles = []

for entity, raw_path in RAW_FILES.items():
    raw_df = read_raw_csv(raw_path)
    bronze_df = add_bronze_metadata(
        df=raw_df,
        entity=entity,
        source_path=raw_path,
        ingestion_id=ingestion_id,
    )
    
    output_path = BRONZE_OUTPUTS[entity]
    write_bronze_parquet(bronze_df, output_path)
    
    bronze_tables[entity] = bronze_df
    
    ingestion_manifest_rows.append(
        {
            "ingestion_id": ingestion_id,
            "entity": entity,
            "source_file": raw_path.name,
            "source_path": str(raw_path),
            "target_file": output_path.name,
            "target_path": str(output_path),
            "source_layer": "raw",
            "target_layer": "bronze",
            "source_format": "csv",
            "target_format": "parquet",
            "raw_rows": len(raw_df),
            "bronze_rows": len(bronze_df),
            "raw_columns": len(raw_df.columns),
            "bronze_columns": len(bronze_df.columns),
            "ingestion_date": STUDY_DATE,
            "notebook": NOTEBOOK_NAME,
        }
    )
    
    reconciliation_rows.append(
        {
            "entity": entity,
            "raw_rows": len(raw_df),
            "bronze_rows": len(bronze_df),
            "row_count_match": len(raw_df) == len(bronze_df),
            "raw_columns": len(raw_df.columns),
            "bronze_columns": len(bronze_df.columns),
            "metadata_columns_added": len(bronze_df.columns) - len(raw_df.columns),
        }
    )
    
    schema_profiles.append(
        pd.DataFrame(
            [
                {
                    "entity": entity,
                    "column": col,
                    "dtype": str(bronze_df[col].dtype),
                    "is_metadata_column": col.startswith("bronze_"),
                }
                for col in bronze_df.columns
            ]
        )
    )
    
    quality_profiles.append(profile_dataframe(entity, bronze_df))

bronze_layer_manifest = pd.DataFrame(ingestion_manifest_rows)
raw_to_bronze_reconciliation = pd.DataFrame(reconciliation_rows)
bronze_schema_summary = pd.concat(schema_profiles, ignore_index=True)
bronze_quality_profile = pd.concat(quality_profiles, ignore_index=True)

bronze_layer_manifest


,ingestion_id,entity,source_file,source_path,target_file,target_path,source_layer,target_layer,source_format,target_format,raw_rows,bronze_rows,raw_columns,bronze_columns,ingestion_date,notebook
0,INGEST_20260429_002250_46935751,customers,customers.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\customers.csv,bronze_customers.parquet,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\bronze\bronze_customers.parquet,raw,bronze,csv,parquet,501,501,11,22,2026-04-29,03_raw_to_bronze_ingestion
1,INGEST_20260429_002250_46935751,contracts,contracts.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\contracts.csv,bronze_contracts.parquet,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\bronze\bronze_contracts.parquet,raw,bronze,csv,parquet,622,622,10,21,2026-04-29,03_raw_to_bronze_ingestion
2,INGEST_20260429_002250_46935751,payments,payments.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\payments.csv,bronze_payments.parquet,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\bronze\bronze_payments.parquet,raw,bronze,csv,parquet,12919,12919,11,22,2026-04-29,03_raw_to_bronze_ingestion
3,INGEST_20260429_002250_46935751,credit_events,credit_events.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\credit_events.csv,bronze_credit_events.parquet,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\bronze\bronze_credit_events.parquet,raw,bronze,csv,parquet,580,580,8,19,2026-04-29,03_raw_to_bronze_ingestion


In [10]:
raw_to_bronze_reconciliation

,entity,raw_rows,bronze_rows,row_count_match,raw_columns,bronze_columns,metadata_columns_added
0,customers,501,501,True,11,22,11
1,contracts,622,622,True,10,21,11
2,payments,12919,12919,True,11,22,11
3,credit_events,580,580,True,8,19,11


In [11]:
if not raw_to_bronze_reconciliation["row_count_match"].all():
    raise ValueError("Existe divergência de linhas entre Raw e Bronze.")

print("Reconciliação Raw → Bronze concluída com sucesso.")

Reconciliação Raw → Bronze concluída com sucesso.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 5. Inspeção das tabelas Bronze</strong></font>

<font size="2" color='#66666'>

>- Vamos visualizar uma amostra das tabelas Bronze geradas.

>- O objetivo é confirmar que os dados originais foram preservados e que os metadados técnicos foram adicionados corretamente.

---

</font></div>

In [12]:
bronze_tables["customers"].head()

,bronze_record_id,customer_id,customer_name,age,monthly_income,employment_type,region,credit_score_base,risk_band_source,signup_date,source_system,extraction_date,bronze_ingestion_id,bronze_ingested_at,bronze_ingestion_date,bronze_source_entity,bronze_source_file,bronze_source_path,bronze_source_layer,bronze_target_layer,bronze_pipeline_step,bronze_row_hash
0,CUSTOMERS_00000001,C000001,Customer 0001,46,7501.58,public_servant,SP,594,high,2025-03-04,crm_core,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:50,2026-04-29,customers,customers.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\customers.csv,raw,bronze,03_raw_to_bronze_ingestion,12077485295847784932
1,CUSTOMERS_00000002,C000002,Customer 0002,34,500.00,formal_employee,RJ,579,high,2019-04-02,crm_core,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:50,2026-04-29,customers,customers.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\customers.csv,raw,bronze,03_raw_to_bronze_ingestion,10111629668991760991
2,CUSTOMERS_00000003,C000003,Customer 0003,58,4725.87,formal_employee,SP,592,high,2018-04-13,crm_core,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:50,2026-04-29,customers,customers.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\customers.csv,raw,bronze,03_raw_to_bronze_ingestion,12439174029935364869
3,CUSTOMERS_00000004,C000004,Customer 0004,52,5666.21,self_employed,PR,686,medium,2021-01-31,crm_core,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:50,2026-04-29,customers,customers.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\customers.csv,raw,bronze,03_raw_to_bronze_ingestion,6776480882729100784
4,CUSTOMERS_00000005,C000005,Customer 0005,43,6921.51,self_employed,SP,723,low,2020-09-30,crm_core,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:50,2026-04-29,customers,customers.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\customers.csv,raw,bronze,03_raw_to_bronze_ingestion,17888665693173395794


In [13]:
bronze_tables["contracts"].head()

,bronze_record_id,contract_id,customer_id,product_type,principal_amount,interest_rate_monthly,term_months,start_date,contract_status,source_system,extraction_date,bronze_ingestion_id,bronze_ingested_at,bronze_ingestion_date,bronze_source_entity,bronze_source_file,bronze_source_path,bronze_source_layer,bronze_target_layer,bronze_pipeline_step,bronze_row_hash
0,CONTRACTS_00000001,CTR000001,C000001,credit_card,2863.83,0.0844,24,2022-12-05,active,loan_management,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,contracts,contracts.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\contracts.csv,raw,bronze,03_raw_to_bronze_ingestion,15603678133230421054
1,CONTRACTS_00000002,CTR000002,C000002,auto_loan,3431.72,0.0225,48,2022-01-01,closed,loan_management,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,contracts,contracts.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\contracts.csv,raw,bronze,03_raw_to_bronze_ingestion,12891029801957839197
2,CONTRACTS_00000003,CTR000003,C000002,credit_card,232.56,0.0538,24,2022-12-23,active,loan_management,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,contracts,contracts.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\contracts.csv,raw,bronze,03_raw_to_bronze_ingestion,190625083188719845
3,CONTRACTS_00000004,CTR000004,C000003,credit_card,4453.18,0.0637,24,2020-06-05,closed,loan_management,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,contracts,contracts.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\contracts.csv,raw,bronze,03_raw_to_bronze_ingestion,3783499994038012483
4,CONTRACTS_00000005,CTR000005,C000003,payroll_loan,15677.32,0.0193,24,2021-03-20,NaN,loan_management,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,contracts,contracts.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\contracts.csv,raw,bronze,03_raw_to_bronze_ingestion,8915226707712989420


In [14]:
bronze_tables["payments"].head()

,bronze_record_id,payment_id,contract_id,installment_number,due_date,payment_date,scheduled_amount,amount_paid,days_late,payment_status,source_system,extraction_date,bronze_ingestion_id,bronze_ingested_at,bronze_ingestion_date,bronze_source_entity,bronze_source_file,bronze_source_path,bronze_source_layer,bronze_target_layer,bronze_pipeline_step,bronze_row_hash
0,PAYMENTS_00000001,PAY000001,CTR000001,1,2023-01-04,2023-02-03,361.03,361.03,30,late,payment_gateway,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,payments,payments.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\payments.csv,raw,bronze,03_raw_to_bronze_ingestion,8976646003512106211
1,PAYMENTS_00000002,PAY000002,CTR000001,2,2023-02-03,2023-02-03,361.03,361.03,0,paid,payment_gateway,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,payments,payments.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\payments.csv,raw,bronze,03_raw_to_bronze_ingestion,12327911590405780350
2,PAYMENTS_00000003,PAY000003,CTR000001,3,2023-03-05,2023-03-06,361.03,361.03,1,paid,payment_gateway,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,payments,payments.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\payments.csv,raw,bronze,03_raw_to_bronze_ingestion,7968094162022333954
3,PAYMENTS_00000004,PAY000004,CTR000001,4,2023-04-04,2023-06-03,361.03,361.03,60,late,payment_gateway,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,payments,payments.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\payments.csv,raw,bronze,03_raw_to_bronze_ingestion,260922600177131241
4,PAYMENTS_00000005,PAY000005,CTR000001,5,2023-05-04,2023-05-05,361.03,361.03,1,paid,payment_gateway,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,payments,payments.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\payments.csv,raw,bronze,03_raw_to_bronze_ingestion,1844386861033534335


In [15]:
bronze_tables["credit_events"].head()

,bronze_record_id,event_id,customer_id,contract_id,event_type,event_date,event_value,source_system,extraction_date,bronze_ingestion_id,bronze_ingested_at,bronze_ingestion_date,bronze_source_entity,bronze_source_file,bronze_source_path,bronze_source_layer,bronze_target_layer,bronze_pipeline_step,bronze_row_hash
0,CREDIT_EVENTS_00000001,EVT000001,C000001,CTR000001,collection_contact,2024-03-27,1.0,credit_operations,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,credit_events,credit_events.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\credit_events.csv,raw,bronze,03_raw_to_bronze_ingestion,5895653666407011672
1,CREDIT_EVENTS_00000002,EVT000002,C000001,CTR000001,bureau_update,2025-07-27,886.0,credit_operations,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,credit_events,credit_events.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\credit_events.csv,raw,bronze,03_raw_to_bronze_ingestion,10071004017250543449
2,CREDIT_EVENTS_00000003,EVT000003,C000002,CTR000002,delinquency_alert,2022-02-21,30.0,credit_operations,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,credit_events,credit_events.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\credit_events.csv,raw,bronze,03_raw_to_bronze_ingestion,3918307231293618057
3,CREDIT_EVENTS_00000004,EVT000004,C000002,CTR000002,collection_contact,2025-06-22,2.0,credit_operations,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,credit_events,credit_events.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\credit_events.csv,raw,bronze,03_raw_to_bronze_ingestion,12737478405588504166
4,CREDIT_EVENTS_00000005,EVT000005,C000002,CTR000003,bureau_update,2024-10-24,444.0,credit_operations,2026-04-28,INGEST_20260429_002250_46935751,2026-04-29T00:22:52,2026-04-29,credit_events,credit_events.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\credit_events.csv,raw,bronze,03_raw_to_bronze_ingestion,3869792689150217887


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 6. Perfil de qualidade da Bronze</strong></font>

<font size="2" color='#66666'>

>- A camada Bronze não corrige problemas de negócio, mas precisa registrar o estado dos dados ingeridos.

>- Nesta etapa, avaliamos:
    - nulos;
    - blanks;
    - tipos inferidos;
    - amostras de valores;
    - colunas técnicas adicionadas.

---

</font></div>

In [16]:
bronze_quality_profile.head(30)

,entity,column,total_rows,null_count,blank_count,null_pct,blank_pct,dtype,sample_values
0,customers,bronze_record_id,501,0,0,0.0000,0.0,str,"CUSTOMERS_00000001, CUSTOMERS_00000002, CUSTOMERS_00000003"
1,customers,customer_id,501,0,0,0.0000,0.0,str,"C000001, C000002, C000003"
2,customers,customer_name,501,0,0,0.0000,0.0,str,"Customer 0001, Customer 0002, Customer 0003"
3,customers,age,501,0,0,0.0000,0.0,int64,"46, 34, 58"
4,customers,monthly_income,501,15,0,0.0299,0.0,float64,"7501.58, 500.0, 4725.87"
5,customers,employment_type,501,0,0,0.0000,0.0,str,"public_servant, formal_employee, formal_employee"
6,customers,region,501,0,0,0.0000,0.0,str,"SP, RJ, SP"
7,customers,credit_score_base,501,0,0,0.0000,0.0,int64,"594, 579, 592"
8,customers,risk_band_source,501,0,0,0.0000,0.0,str,"high, high, high"
9,customers,signup_date,501,0,0,0.0000,0.0,str,"2025-03-04, 2019-04-02, 2018-04-13"


In [17]:
bronze_entity_quality_summary = pd.DataFrame(
    [
        {
            "entity": entity,
            "rows": len(df),
            "columns": len(df.columns),
            "duplicated_rows": int(df.duplicated().sum()),
            "duplicated_hashes": int(df["bronze_row_hash"].duplicated().sum()),
            "total_null_cells": int(df.isna().sum().sum()),
            "metadata_columns": len([col for col in df.columns if col.startswith("bronze_")]),
        }
        for entity, df in bronze_tables.items()
    ]
)

bronze_entity_quality_summary


,entity,rows,columns,duplicated_rows,duplicated_hashes,total_null_cells,metadata_columns
0,customers,501,22,0,1,15,11
1,contracts,622,21,0,1,6,11
2,payments,12919,22,0,1,378,11
3,credit_events,580,19,0,0,6,11


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 7. Leitura técnica da ingestão</strong></font>

<font size="2" color='#66666'>

>- A ingestão Raw → Bronze foi realizada sem aplicar regras de limpeza de negócio.

>- Isso significa que:
    - duplicidades originadas na Raw foram preservadas;
    - valores ausentes foram preservados;
    - inconsistências serão tratadas posteriormente;
    - metadados técnicos foram adicionados para rastreabilidade.

>- Essa decisão é intencional em uma arquitetura Lakehouse, pois a Bronze deve funcionar como uma camada de aterrissagem estruturada e auditável.

---

</font></div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 8. Validação dos arquivos Parquet gerados</strong></font>

<font size="2" color='#66666'>

>- Agora vamos verificar se os arquivos Parquet foram criados corretamente na camada Bronze.

---

</font></div>

In [18]:
bronze_file_validation = pd.DataFrame(
    [
        {
            "entity": entity,
            "bronze_file": str(path),
            "exists": path.exists(),
            "file_size_bytes": path.stat().st_size if path.exists() else 0,
        }
        for entity, path in BRONZE_OUTPUTS.items()
    ]
)

bronze_file_validation


,entity,bronze_file,exists,file_size_bytes
0,customers,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\bronze\bronze_customers.parquet,True,42314
1,contracts,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\bronze\bronze_contracts.parquet,True,49384
2,payments,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\bronze\bronze_payments.parquet,True,521789
3,credit_events,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\bronze\bronze_credit_events.parquet,True,41398


In [19]:
missing_bronze_files = bronze_file_validation.loc[~bronze_file_validation["exists"], "bronze_file"].tolist()

if missing_bronze_files:
    raise FileNotFoundError(f"Arquivos Bronze não encontrados: {missing_bronze_files}")

print("Todos os arquivos Bronze foram criados com sucesso.")


Todos os arquivos Bronze foram criados com sucesso.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 9. Teste de leitura dos arquivos Bronze</strong></font>

<font size="2" color='#66666'>

>- Para garantir que os Parquets estão utilizáveis, vamos reler os arquivos gerados.

---

</font></div>

In [20]:
bronze_readback_summary = []

for entity, output_path in BRONZE_OUTPUTS.items():
    df_readback = pd.read_parquet(output_path)
    
    bronze_readback_summary.append(
        {
            "entity": entity,
            "rows_read": len(df_readback),
            "columns_read": len(df_readback.columns),
            "read_success": True,
        }
    )

bronze_readback_summary = pd.DataFrame(bronze_readback_summary)

bronze_readback_summary


,entity,rows_read,columns_read,read_success
0,customers,501,22,True
1,contracts,622,21,True
2,payments,12919,22,True
3,credit_events,580,19,True


In [21]:
bronze_validation_summary = raw_to_bronze_reconciliation.merge(
    bronze_readback_summary,
    on="entity",
    how="left",
)

bronze_validation_summary["bronze_rows_match_readback"] = (
    bronze_validation_summary["bronze_rows"] == bronze_validation_summary["rows_read"]
)

bronze_validation_summary


,entity,raw_rows,bronze_rows,row_count_match,raw_columns,bronze_columns,metadata_columns_added,rows_read,columns_read,read_success,bronze_rows_match_readback
0,customers,501,501,True,11,22,11,501,22,True,True
1,contracts,622,622,True,10,21,11,622,21,True,True
2,payments,12919,12919,True,11,22,11,12919,22,True,True
3,credit_events,580,580,True,8,19,11,580,19,True,True


In [22]:
if not bronze_validation_summary["bronze_rows_match_readback"].all():
    raise ValueError("Existe divergência entre escrita e leitura dos arquivos Bronze.")

print("Validação de leitura dos Parquets concluída com sucesso.")


Validação de leitura dos Parquets concluída com sucesso.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 10. Evidências de aprendizagem para o KG</strong></font>

<font size="2" color='#66666'>

>- Este notebook gera evidência real para o projeto `databricks-learning-kg`.

>- A evidência agora está diretamente conectada a tópicos de Engenharia de Dados, como:
    - Data Ingestion;
    - File Formats;
    - Schema Definition;
    - Pipeline Idempotency;
    - Medallion Architecture;
    - Data Quality.

---

</font></div>

In [23]:
learning_evidence = pd.DataFrame(
    [
        {
            "evidence_id": "EVID_CR_LH_012",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Data Ingestion",
            "evidence_type": "raw_to_bronze_ingestion",
            "confidence_delta_suggested": 0.15,
            "mastery_delta_suggested": 0.10,
            "evidence_description": "Ingestão dos arquivos CSV da camada Raw para a camada Bronze com metadados técnicos.",
        },
        {
            "evidence_id": "EVID_CR_LH_013",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "File Formats",
            "evidence_type": "csv_to_parquet_conversion",
            "confidence_delta_suggested": 0.15,
            "mastery_delta_suggested": 0.10,
            "evidence_description": "Conversão dos arquivos CSV de origem para Parquet na camada Bronze.",
        },
        {
            "evidence_id": "EVID_CR_LH_014",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Schema Definition",
            "evidence_type": "bronze_schema_tracking",
            "confidence_delta_suggested": 0.10,
            "mastery_delta_suggested": 0.05,
            "evidence_description": "Registro de schema e tipos inferidos para as tabelas Bronze.",
        },
        {
            "evidence_id": "EVID_CR_LH_015",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Pipeline Idempotency",
            "evidence_type": "idempotent_overwrite",
            "confidence_delta_suggested": 0.10,
            "mastery_delta_suggested": 0.10,
            "evidence_description": "Persistência Bronze com sobrescrita controlada, permitindo reexecução previsível do pipeline.",
        },
        {
            "evidence_id": "EVID_CR_LH_016",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Medallion Architecture",
            "evidence_type": "bronze_layer_implementation",
            "confidence_delta_suggested": 0.10,
            "mastery_delta_suggested": 0.10,
            "evidence_description": "Implementação prática da transição Raw para Bronze dentro da Arquitetura Medalhão.",
        },
        {
            "evidence_id": "EVID_CR_LH_017",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Data Quality",
            "evidence_type": "bronze_quality_profile",
            "confidence_delta_suggested": 0.10,
            "mastery_delta_suggested": 0.05,
            "evidence_description": "Geração de perfil de qualidade para a camada Bronze sem aplicar limpeza de negócio.",
        },
    ]
)

learning_evidence


,evidence_id,project,notebook,evidence_date,topic,evidence_type,confidence_delta_suggested,mastery_delta_suggested,evidence_description
0,EVID_CR_LH_012,Databricks Credit Risk Lakehouse,03_raw_to_bronze_ingestion,2026-04-29,Data Ingestion,raw_to_bronze_ingestion,0.15,0.10,Ingestão dos arquivos CSV da camada Raw para a camada Bronze com metadados técnicos.
1,EVID_CR_LH_013,Databricks Credit Risk Lakehouse,03_raw_to_bronze_ingestion,2026-04-29,File Formats,csv_to_parquet_conversion,0.15,0.10,Conversão dos arquivos CSV de origem para Parquet na camada Bronze.
2,EVID_CR_LH_014,Databricks Credit Risk Lakehouse,03_raw_to_bronze_ingestion,2026-04-29,Schema Definition,bronze_schema_tracking,0.10,0.05,Registro de schema e tipos inferidos para as tabelas Bronze.
3,EVID_CR_LH_015,Databricks Credit Risk Lakehouse,03_raw_to_bronze_ingestion,2026-04-29,Pipeline Idempotency,idempotent_overwrite,0.10,0.10,"Persistência Bronze com sobrescrita controlada, permitindo reexecução previsível do pipeline."
4,EVID_CR_LH_016,Databricks Credit Risk Lakehouse,03_raw_to_bronze_ingestion,2026-04-29,Medallion Architecture,bronze_layer_implementation,0.10,0.10,Implementação prática da transição Raw para Bronze dentro da Arquitetura Medalhão.
5,EVID_CR_LH_017,Databricks Credit Risk Lakehouse,03_raw_to_bronze_ingestion,2026-04-29,Data Quality,bronze_quality_profile,0.10,0.05,Geração de perfil de qualidade para a camada Bronze sem aplicar limpeza de negócio.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 11. Exportação dos artefatos</strong></font>

<font size="2" color='#66666'>

>- Agora vamos salvar os artefatos de controle, documentação e evidência.

>- Esses arquivos serão usados para:
    - documentação do projeto;
    - auditoria da ingestão;
    - atualização futura do Knowledge Graph;
    - preparação dos próximos notebooks.

---

</font></div>

In [24]:
bronze_layer_manifest.to_csv(EXPORTS_DIR / "bronze_layer_manifest.csv", index=False, encoding="utf-8")
raw_to_bronze_reconciliation.to_csv(EXPORTS_DIR / "raw_to_bronze_reconciliation.csv", index=False, encoding="utf-8")
bronze_schema_summary.to_csv(EXPORTS_DIR / "bronze_schema_summary.csv", index=False, encoding="utf-8")
bronze_quality_profile.to_csv(EXPORTS_DIR / "bronze_quality_profile.csv", index=False, encoding="utf-8")
bronze_entity_quality_summary.to_csv(EXPORTS_DIR / "bronze_entity_quality_summary.csv", index=False, encoding="utf-8")
bronze_file_validation.to_csv(EXPORTS_DIR / "bronze_file_validation.csv", index=False, encoding="utf-8")
bronze_validation_summary.to_csv(EXPORTS_DIR / "bronze_validation_summary.csv", index=False, encoding="utf-8")
learning_evidence.to_csv(EXPORTS_DIR / "learning_evidence_notebook_03.csv", index=False, encoding="utf-8")

print("Artefatos da camada Bronze exportados com sucesso.")


Artefatos da camada Bronze exportados com sucesso.


In [25]:
summary_report = f"""# Notebook 03 — Raw to Bronze Ingestion

## Projeto

{PROJECT_NAME}

## Data

{STUDY_DATE}

## Objetivo

Realizar a ingestão dos arquivos CSV da camada Raw para a camada Bronze, adicionando metadados técnicos e persistindo os dados em formato Parquet.

## Manifesto da camada Bronze

{df_to_markdown_safe(bronze_layer_manifest[["entity", "source_file", "target_file", "raw_rows", "bronze_rows", "raw_columns", "bronze_columns", "target_format"]])}

## Reconciliação Raw → Bronze

{df_to_markdown_safe(raw_to_bronze_reconciliation)}

## Qualidade inicial da Bronze

{df_to_markdown_safe(bronze_entity_quality_summary)}

## Validação de arquivos Bronze

{df_to_markdown_safe(bronze_file_validation[["entity", "exists", "file_size_bytes"]])}

## Evidências de aprendizagem geradas

{df_to_markdown_safe(learning_evidence[["evidence_id", "topic", "evidence_type", "confidence_delta_suggested", "mastery_delta_suggested"]])}

## Leitura executiva

Este notebook implementou a primeira etapa prática de ingestão do projeto `Databricks Credit Risk Lakehouse`.

Os dados brutos foram preservados e enriquecidos com metadados técnicos, permitindo rastreabilidade entre Raw e Bronze. A persistência em Parquet prepara o projeto para etapas posteriores de limpeza, padronização, integração e construção de features de risco de crédito.

## Próximo passo

O próximo notebook será:

`Notebook 04 — Bronze to Silver Data Cleaning`

Nele, as tabelas Bronze serão tratadas para remoção ou marcação de duplicidades, padronização de tipos, tratamento de nulos, validação de chaves e preparação da camada Silver.
"""

report_path = DOCS_DIR / "notebook_03_raw_to_bronze_ingestion_summary.md"
report_path.write_text(summary_report, encoding="utf-8")

report_path


WindowsPath('D:/_DS-Projects/Data-Science/databricks-credit-risk-lakehouse/docs/notebook_03_raw_to_bronze_ingestion_summary.md')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 12. Conclusão executiva</strong></font>

<font size="2" color='#66666'>

>- Este notebook implementou a ingestão Raw → Bronze do projeto `Databricks Credit Risk Lakehouse`.

>- Foram gerados:
    - arquivos Parquet na camada Bronze;
    - metadados técnicos de ingestão;
    - identificadores técnicos por registro;
    - hashes de rastreabilidade;
    - manifesto da camada Bronze;
    - reconciliação Raw → Bronze;
    - perfil de qualidade da Bronze;
    - validação de escrita e leitura dos arquivos Parquet;
    - evidências reais para atualização futura do Knowledge Graph.

>- **Leitura executiva**
    - A camada Bronze agora representa a primeira materialização estruturada do Lakehouse.
    - Ela preserva os dados próximos da origem, mas acrescenta rastreabilidade técnica suficiente para auditoria, reprocessamento e evolução incremental do pipeline.

>- **Próximo notebook**
    - `Notebook 04 — Bronze to Silver Data Cleaning`
    - O próximo passo será aplicar regras de limpeza, tipagem, deduplicação, padronização e validação para criar a camada Silver.

---

</font></div>